# NB1 · Veriye ulaşmak

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Atölyede ne yapıyoruz

Altı defter boyunca çalışan bir klinik karar destek sistemi yazacağız. Kodu siz
yazmayacaksınız. Her adımda bir istem verilir; bunu bir üretken yapay zekâ aracına
(ChatGPT, Claude, Gemini) verir, aracın ürettiği kodu boş hücreye yapıştırıp
çalıştırırsınız.

| Defter | Eklenen katman |
|---|---|
| NB1 | Veriye ulaşma |
| NB2 | Temizleme, eğitim ve test ayrımı, modele hazırlama |
| NB3 | Model eğitimi, tahmin ve değerlendirme |
| NB4 | Modelin kararını açıklama |
| NB5 | Güvenlik kontrolleri ve uyumluluk raporu |
| NB6 | Web arayüzü |

Her adımdan sonra birkaç soru gelir. Kodunuza ve çıktısına bakarak cevaplayınız.
Atölyenin asıl içeriği bu sorulardır: Üretilen kod ilk bakışta doğru görünür, yanlış
olduğu yerler de öyle görünür.


## Kod defterden deftere taşınır

Yapıştırma hücrelerinin ilk satırında `#@cdss adim_adi` yazar. **Bu satırı silmeyiniz.**
Kodunuzu altına yapıştırınız. Defterin sonunda `kit.topla()` işaretli hücreleri tek blok
hâlinde verir; o bloğu bir sonraki defterin başına yapıştırırsınız.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

DEPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{DEPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır.')


---

## Veri kümesi seçimi

Altı veri kümesi hazırlanmıştır. Hepsi açık erişimlidir ve Colab içinden doğrudan alınır.
Birini seçiniz; sonraki adımlar seçiminize göre ilerler.

| Kod | Tür | Veri | Tahmin edilecek durum |
|---|---|---|---|
| `mimic-icu` | tablo | MIMIC-IV demo, 100 hastanın yoğun bakım kaydı | Yatışın üç günü aşması |
| `wisconsin` | tablo | Breast Cancer Wisconsin, 569 örnek | Kitlenin habis olması |
| `pneumonia-mnist` | görüntü | PneumoniaMNIST, 5.856 göğüs radyografisi | Pnömoni bulunması |
| `breast-mnist` | görüntü | BreastMNIST, 780 ultrasonografi görüntüsü | Kitlenin habis olması |
| `mimic-ecg` | sinyal | MIMIC-IV-ECG demo, 92 hastadan 659 EKG | Yatışın üç günü aşması |
| `synthetic-notes` | metin | Üretilen klinik notlar | Kendi belirlediğiniz durum |

Veri kümesi adları özgün hâliyle bırakılmıştır; literatürde bu adlarla geçerler.

Üç noktayı seçmeden önce bilmekte fayda var. MIMIC yoğun bakım ve EKG kümelerinde bir
hastanın birden fazla kaydı bulunur, dolayısıyla eğitim ve test ayrımı hasta düzeyinde
yapılmak zorundadır. MedMNIST kümelerinde hasta kimliği yoktur. EKG kümesi, klinik
demodaki hastalarla aynı 92 hastayı içerir; o yolda aynı sonuç bu kez sinyalden tahmin
edilir.


---

## Adım 1 · Hazırlık

Program kütüphanelerin içe aktarılmasıyla ve değişmeyecek değerlerin tanımlanmasıyla
başlar. Bu değerler problemi tarif eder ve sonraki bütün adımlar bunları okur.

Seçtiğiniz kümenin istemini kopyalayıp yapay zekâ aracına veriniz.


### İstem · `mimic-icu`

```
Klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre hazırlık bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken kütüphaneleri içe aktar. Sonra şu
değişkenleri tanımla:

VERI_SETI = 'mimic-icu'
PROBLEM = 'Yoğun bakıma alınan hastanın üç günden uzun kalıp kalmayacağının öngörülmesi'
KARAR_ANI = 'Yoğun bakıma girişten altı saat sonra'
VERI_KOKU = 'https://physionet.org/files/mimic-iv-demo/2.2'
RASTGELE_TOHUM = 42
KARAR_PENCERESI_SAAT = 6
HEDEF_ESIK_GUN = 3

Kütüphane sürümlerini ve tanımladığın değişkenleri ekrana yazdır. Kod satırlarına kısa
Türkçe yorumlar ekle.
```


### İstem · `wisconsin`

```
Klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre hazırlık bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken kütüphaneleri içe aktar. Sonra şu
değişkenleri tanımla:

VERI_SETI = 'wisconsin'
PROBLEM = 'Meme kitlesinin habis olup olmadığının öngörülmesi'
KARAR_ANI = 'İnce iğne aspirasyon biyopsisi ölçümleri elde edildiğinde'
VERI_KOKU = 'scikit-learn'
RASTGELE_TOHUM = 42

Kütüphane sürümlerini ve tanımladığın değişkenleri ekrana yazdır. Kod satırlarına kısa
Türkçe yorumlar ekle.
```


### İstem · `pneumonia-mnist`

```
Klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre hazırlık bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken kütüphaneleri içe aktar. Sonra şu
değişkenleri tanımla:

VERI_SETI = 'pneumonia-mnist'
PROBLEM = 'Çocuk göğüs radyografisinde pnömoni bulunup bulunmadığının öngörülmesi'
KARAR_ANI = 'Radyografi çekildikten hemen sonra, uzman değerlendirmesinden önce'
VERI_KOKU = 'medmnist'
RASTGELE_TOHUM = 42

Kütüphane sürümlerini ve tanımladığın değişkenleri ekrana yazdır. Kod satırlarına kısa
Türkçe yorumlar ekle.
```


### İstem · `breast-mnist`

```
Klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre hazırlık bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken kütüphaneleri içe aktar. Sonra şu
değişkenleri tanımla:

VERI_SETI = 'breast-mnist'
PROBLEM = 'Ultrasonografide görülen meme kitlesinin habis olup olmadığının öngörülmesi'
KARAR_ANI = 'Ultrasonografi görüntüsü elde edildiğinde'
VERI_KOKU = 'medmnist'
RASTGELE_TOHUM = 42

Kütüphane sürümlerini ve tanımladığın değişkenleri ekrana yazdır. Kod satırlarına kısa
Türkçe yorumlar ekle.
```


### İstem · `mimic-ecg`

```
Klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre hazırlık bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken kütüphaneleri içe aktar. Sonra şu
değişkenleri tanımla:

VERI_SETI = 'mimic-ecg'
PROBLEM = 'Yoğun bakım yatışının üç günü aşıp aşmayacağının EKG üzerinden öngörülmesi'
KARAR_ANI = 'EKG çekildiğinde'
VERI_KOKU = 'https://physionet.org/files/mimic-iv-ecg-demo/0.1'
RASTGELE_TOHUM = 42
HEDEF_ESIK_GUN = 3

Kütüphane sürümlerini ve tanımladığın değişkenleri ekrana yazdır. Kod satırlarına kısa
Türkçe yorumlar ekle.
```


### İstem · `synthetic-notes`

```
Klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre hazırlık bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken kütüphaneleri içe aktar. Sonra şu
değişkenleri tanımla:

VERI_SETI = 'synthetic-notes'
PROBLEM = '[çözmek istediğiniz klinik problemi tek cümleyle yazınız]'
KARAR_ANI = '[sistemin çıktı üreteceği an]'
VERI_KOKU = 'sentetik'
RASTGELE_TOHUM = 42

Kütüphane sürümlerini ve tanımladığın değişkenleri ekrana yazdır. Kod satırlarına kısa
Türkçe yorumlar ekle.
```


In [ ]:
#@cdss hazirlik
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kodunuza bakın

- Hangi kütüphaneler içe aktarıldı? Her birinin ne işe yaradığını söyleyebiliyor musunuz?
- Rastgele tohum tanımlanmasaydı ne değişirdi?
- Değişkenler fonksiyonların dışında, dosyanın başında mı duruyor? Klinik bir sistemde
  eşik değerlerinin tek yerde toplanması neden önemli?


---

## Adım 2 · Veriyi yükleme

Veriyi programa alan fonksiyonu yazdıracaksınız. Her istem aynı yapıda bir sonuç üretir:
`hasta_id` ve `hedef` sütunlarını içeren bir veri çerçevesi. Bu ortak yapı sayesinde
sonraki adımlar veri türünden bağımsız ilerler.

Dosya adreslerini ve sütun adlarını isteme ben yazdım. Yapay zekâ aracı bunları bilmez;
bilmediğini tahmin eder ve tahmini genellikle yanlış olur.


### İstem · `mimic-icu`

```
Yukarıdaki değişkenleri kullanarak veriyi yükleyen bir fonksiyon yaz. Adı veri_yukle
olsun.

Üç dosya gerekli, hepsi sıkıştırılmış CSV:
  {VERI_KOKU}/icu/icustays.csv.gz     subject_id, hadm_id, stay_id, first_careunit,
                                      intime, outtime, los
  {VERI_KOKU}/hosp/patients.csv.gz    subject_id, gender, anchor_age
  {VERI_KOKU}/hosp/admissions.csv.gz  subject_id, hadm_id, admission_type, insurance

Yoğun bakım yatışlarından başla. los değeri KARAR_PENCERESI_SAAT saatten küçük olan
yatışları çıkar. hedef sütununu üret: los değeri HEDEF_ESIK_GUN gününü aşıyorsa 1,
aşmıyorsa 0. subject_id sütununun adını hasta_id yap. Diğer iki dosyadan demografik
bilgileri ekle; satır sayısı değişmemeli.

los ve outtime sütunlarını sonuçtan çıkar. Bu iki değer hasta taburcu olduktan sonra
bilinir, karar anında elimizde yoktur.

Fonksiyonu çağır, sonucu kohort değişkeninde tut, ilk beş satırı göster.
```


### İstem · `wisconsin`

```
Breast Cancer Wisconsin veri kümesini yükleyen bir fonksiyon yaz. Adı veri_yukle olsun.
Bu küme scikit-learn içinde hazır gelir, load_breast_cancer ile açılır.

Kümeyi bir veri çerçevesine çevir. hedef sütununu üret: Kitle habis ise 1, benign ise 0.
Her satır ayrı bir kişiye ait olduğu için hasta_id sütununu satır sırasından üret ve
bunun gerçek bir hasta kimliği olmadığını yorum satırında belirt.

Fonksiyonu çağır, sonucu kohort değişkeninde tut, ilk beş satırı göster.
```


### İstem · `pneumonia-mnist` ve `breast-mnist`

```
MedMNIST koleksiyonundan bir görüntü kümesini yükleyen bir fonksiyon yaz. Adı veri_yukle
olsun. Kurulum: pip install medmnist

VERI_SETI değeri 'pneumonia-mnist' ise PneumoniaMNIST, 'breast-mnist' ise BreastMNIST
kullan. Görüntüler 28x28 ve tek kanallıdır. Küme eğitim, doğrulama ve test olarak bölünmüş
gelir; üçünü birleştir, bölmeyi biz NB2'de yapacağız.

Her görüntü bir satır olacak biçimde bir veri çerçevesi kur: goruntu ve hedef sütunları.
Bu kümede hasta kimliği yok; hasta_id sütununu satır sırasından üret ve bunun gerçek bir
kimlik olmadığını yorum satırında belirt. Küme büyükse ilk 2000 görüntüyle sınırla.

Fonksiyonu çağır, sonucu kohort değişkeninde tut, sınıf dağılımını yazdır.
```


### İstem · `mimic-ecg`

```
EKG kayıtlarını yükleyen ve hedefi klinik kayıttan getiren bir fonksiyon yaz. Adı
veri_yukle olsun. Kurulum: pip install wfdb

EKG kayıtları {VERI_KOKU} adresinde, WFDB biçiminde, on saniyelik ve 500 Hz. Kayıt listesi
record_list.csv dosyasında. Her hastanın kayıtları kendi klasöründe ve klasör adı hasta
kimliğidir.

record_list.csv dosyasını oku ve ilk 200 kaydı al. Her kaydı wfdb ile oku, yalnızca
birinci derivasyonu sakla.

Hedefi şu adresteki klinik kayıttan getir:
  https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz  (subject_id, los)
Her hasta için en uzun yatışı bul; HEDEF_ESIK_GUN gününü aşıyorsa hedef 1, aşmıyorsa 0.
Yoğun bakım kaydı bulunmayan hastaları çıkar.

Sonuçta hasta_id, sinyal ve hedef sütunları olsun. Bir hastanın birden fazla kaydı
olabileceğini ekrana yazdır. Fonksiyonu çağır, sonucu kohort değişkeninde tut.
```


### İstem · `synthetic-notes`

```
Sentetik klinik not üreten bir fonksiyon yaz. Adı veri_yukle olsun.

Notlar Türkçe olsun ve gerçek klinik notları zorlaştıran özellikleri taşısın:
kısaltmalar, olumsuzlama, belirsizlik ifadeleri, tekrarlayan şablon cümleler ve önceki
nottan kopyalanmış bölümler. PROBLEM ve KARAR_ANI değişkenlerinde tarif ettiğim duruma
göre üret. Aradığımız durum tek bir kelimeden anlaşılmasın.

Sonuçta hasta_id, metin ve hedef sütunları olsun; aynı hastanın birden fazla notu
bulunsun. Fonksiyonu çağır, sonucu kohort değişkeninde tut, üç örnek notu yazdır.
```


In [ ]:
#@cdss veri_yukleme
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Veriye bakın

Aşağıdaki hücre hazır gelir, değiştirmeden çalıştırınız.


In [ ]:
print('Satır sayısı :', len(kohort))
print('Sütun sayısı :', kohort.shape[1])
print('Hasta sayısı :', kohort['hasta_id'].nunique())
print('Hedef oranı  :', f"{kohort['hedef'].mean():.1%}")
print()
print('Sütunlar:', list(kohort.columns))
print()
print('Eksik veri oranı en yüksek beş sütun:')
print(kohort.isna().mean().sort_values(ascending=False).head(5).round(3).to_string())


### Kodunuza bakın

- Hasta sayısı satır sayısından küçük mü? Küçükse bunun eğitim ve test ayrımı için ne
  anlama geldiğini düşününüz; NB2'de bu konuya döneceğiz.
- Hedef oranı nedir? Dengesizse hangi başarım ölçütü yanıltıcı olur?
- Kodda hangi sütunlar bilerek çıkarıldı? Neden çıkarıldıklarını açıklayabiliyor musunuz?
- Kalan sütunlar arasında karar anında elinizde olmayacak bir bilgi var mı? Varsa
  istemi düzeltip kodu yeniden ürettiriniz.

Son soru bu defterin en önemli sorusudur. Karar anında mevcut olmayan bir bilgi modele
girerse model kusursuza yakın başarım gösterir ve sahada hiç çalışmaz. Buna veri sızıntısı
denir; hata vermez, sessizce ilerler.


---

## Defter sonu · Kodu toplayın

Aşağıdaki hücre bu defterde yazdıklarınızı tek blok hâlinde verir. Bloğu kopyalayıp NB2
defterinin ilk hücresine yapıştıracaksınız. Hücre ayrıca dosyayı bilgisayarınıza
indirir; indirme başlamazsa soldaki dosya panelinden alabilirsiniz.


In [ ]:
kod = kit.topla('cdss_nb1.py')


## Bu defterde ne yapıldı

Sistem artık hazırlığını yapıyor ve veriyi yüklüyor. İki alışkanlık edinildi: Verinin
neyi içerdiği araca açıkça söylendi ve karar anında mevcut olmayan bilgiler daha yükleme
sırasında çıkarıldı.

NB2'de temizleme, eğitim ile test ayrımı ve modele hazırlama eklenecek.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
Kullandığınız veri kümesi öğretim için hazırlanmış açık bir kümedir ve kendi kurumunuzun
hasta popülasyonunu temsil etmez.
